In [76]:
import pandas as pd

consultations = pd.read_csv("data/consultations.csv")
doctors = pd.read_csv("data/doctors.csv")
patients = pd.read_csv("data/patients.csv")
services = pd.read_csv("data/services.csv")
time = pd.read_csv("data/time.csv")

In [77]:
time

,ID_Temps,Date,Année,Mois,Trimestre,Jour_Semaine
0,1,2024-01-01,2024,1,1,Lundi
1,2,2024-01-02,2024,1,1,Mardi
2,3,2024-01-03,2024,1,1,Mercredi
3,4,2024-01-04,2024,1,1,Jeudi
4,5,2024-01-05,2024,1,1,Vendredi
...,...,...,...,...,...,...
360,361,2024-12-26,2024,12,4,Jeudi
361,362,2024-12-27,2024,12,4,Vendredi
362,363,2024-12-28,2024,12,4,Samedi
363,364,2024-12-29,2024,12,4,Dimanche


## Dimension date 

In [78]:
H_annee = time[['Année']].drop_duplicates().reset_index(drop=True)
H_annee['id_annee'] = H_annee.index + 1
H_annee

,Année,id_annee
0,2024,1


In [79]:
H_trimestre = time[['Trimestre','Année']].drop_duplicates()
H_trimestre = H_trimestre.merge(H_annee,on='Année')
H_trimestre=H_trimestre.drop(columns=['Année'])
H_trimestre['id_trimestre'] = H_trimestre.index + 1
H_trimestre=H_trimestre[['id_trimestre','Trimestre','id_annee']]
H_trimestre

,id_trimestre,Trimestre,id_annee
0,1,1,1
1,2,2,1
2,3,3,1
3,4,4,1


In [80]:
H_mois = time[['Mois','Trimestre']].drop_duplicates()
H_mois = H_mois.merge(H_trimestre,on='Trimestre')
H_mois = H_mois.drop(columns=['Trimestre'])
H_mois['id_mois'] = H_mois.index + 1
H_mois = H_mois[['id_mois','Mois','id_trimestre']]
H_mois

,id_mois,Mois,id_trimestre
0,1,1,1
1,2,2,1
2,3,3,1
3,4,4,2
4,5,5,2
5,6,6,2
6,7,7,3
7,8,8,3
8,9,9,3
9,10,10,4


In [81]:
D_date = time[['ID_Temps','Date','Jour_Semaine','Mois']]
D_date = D_date.merge(H_mois[['Mois','id_mois']],on='Mois')
D_date = D_date.drop(columns=['Mois'])
D_date

,ID_Temps,Date,Jour_Semaine,id_mois
0,1,2024-01-01,Lundi,1
1,2,2024-01-02,Mardi,1
2,3,2024-01-03,Mercredi,1
3,4,2024-01-04,Jeudi,1
4,5,2024-01-05,Vendredi,1
...,...,...,...,...
360,361,2024-12-26,Jeudi,12
361,362,2024-12-27,Vendredi,12
362,363,2024-12-28,Samedi,12
363,364,2024-12-29,Dimanche,12


### Correction :renommer pour enlever les accents

In [82]:
time = time.rename(columns={
    'Année': 'annee',
    'Mois': 'mois',
    'Trimestre': 'trimestre',
    'Jour_Semaine': 'jour_semaine',
    'Date': 'date',
    'ID_Temps': 'id_temps'
})

In [83]:
H_annee = H_annee.rename(columns={
    'Année': 'annee'
})

In [84]:
H_mois = H_mois.rename(columns={
    'Mois': 'mois'
})

In [85]:
D_date = D_date.rename(columns={
    'ID_Temps': 'id_temps',
    'Date': 'date',
    'Jour_Semaine': 'jour_semaine'
})

## Dimension Patient

In [86]:
H_ville = patients[['Ville']].drop_duplicates().reset_index(drop=True)
H_ville['id_ville'] = H_ville.index + 1

H_ville = H_ville[['id_ville','Ville']]
H_ville

,id_ville,Ville
0,1,Port Rebeccafort
1,2,Aaronburgh
2,3,Meganhaven
3,4,Hudsonchester
4,5,Brownbury
...,...,...
957,958,East Michael
958,959,Rodneymouth
959,960,Christophershire
960,961,Lake Melanie


In [87]:
H_code_postal = patients[['Code_Postal','Ville']].drop_duplicates()

H_code_postal = H_code_postal.merge(H_ville,on='Ville')

H_code_postal['id_code_postal'] = H_code_postal.index + 1

H_code_postal = H_code_postal[['id_code_postal','Code_Postal','id_ville']]
H_code_postal

,id_code_postal,Code_Postal,id_ville
0,1,8333,1
1,2,46687,2
2,3,52369,3
3,4,45515,4
4,5,91295,5
...,...,...,...
995,996,99554,958
996,997,31089,959
997,998,41266,960
998,999,3010,961


In [90]:
D_patient = patients[['ID_Patient','Nom','Prénom','Date_Naissance','Genre','Adresse','Code_Postal']]
D_patient = D_patient.merge(H_code_postal[['Code_Postal','id_code_postal']], on='Code_Postal')
D_patient = D_patient.drop(columns=['Code_Postal'])
D_patient


,ID_Patient,Nom,Prénom,Date_Naissance,Genre,Adresse,id_code_postal
0,1,Dixon,Joseph,1963-03-31,Homme,959 Brittany Circle,1
1,2,Mclaughlin,Robert,1935-11-04,Femme,7450 Hannah Rest Apt. 565,2
2,3,Doyle,Mallory,2008-07-13,Homme,2995 Kelli Viaduct Suite 576,3
3,4,Miller,Kyle,1929-10-04,Femme,0742 Krueger Roads Suite 055,4
4,5,Thomas,Sandra,1974-02-18,Femme,5522 Lawrence Square,5
...,...,...,...,...,...,...,...
997,996,Gomez,Jessica,1942-03-05,Femme,939 Michael Spring,996
998,997,Farmer,Mary,1925-02-23,Homme,3117 Benjamin Drive Apt. 739,997
999,998,Shaw,Susan,1930-02-18,Homme,70653 Anderson Lane Apt. 739,998
1000,999,Mcintosh,Bruce,1964-01-16,Femme,823 Myers Neck,999


In [91]:
D_patient = D_patient.rename(columns={
    'ID_Patient':'id_patient',
    'Nom':'nom_patient',
    'Prénom':'prenom_patient',
    'Date_Naissance':'date_naissance',
    'Genre':'genre',
    'Adresse':'adresse'
})

In [92]:
H_ville = H_ville.rename(columns={'Ville':'ville'})
H_code_postal = H_code_postal.rename(columns={'Code_Postal':'code_postal'})

## Dimension Docteur

In [98]:
H_specialite = doctors[['Spécialité']].drop_duplicates().reset_index(drop=True)
H_specialite['id_specialite'] = H_specialite.index + 1
H_specialite = H_specialite[['id_specialite','Spécialité']]
H_specialite

,id_specialite,Spécialité
0,1,Généraliste
1,2,Cardiologie
2,3,Radiologie
3,4,Pédiatrie
4,5,Dermatologie


In [102]:
D_doctor = doctors[['ID_Docteur', 'Nom', 'Prénom', 'Ancienneté', 'Spécialité']]
D_doctor = D_doctor.merge(H_specialite[['Spécialité','id_specialite']], on='Spécialité')
D_doctor = D_doctor.drop(columns=['Spécialité'])
D_doctor

,ID_Docteur,Nom,Prénom,Ancienneté,id_specialite
0,1,Savage,Elizabeth,18,1
1,2,Camacho,Eddie,15,2
2,3,Kennedy,Amanda,39,2
3,4,Wilson,Keith,15,3
4,5,Holder,Kristy,11,2
...,...,...,...,...,...
95,96,Anderson,Richard,27,4
96,97,Robinson,Sabrina,39,4
97,98,Ortiz,Jonathan,23,2
98,99,Peters,Alan,40,5


In [104]:
D_doctor = D_doctor.rename(columns={
    'ID_Docteur':'id_docteur',
    'Nom':'nom_docteur',
    'Prénom':'prenom_docteur',
    'Ancienneté':'anciennete'
})
D_doctor

,id_docteur,nom_docteur,prenom_docteur,anciennete,id_specialite
0,1,Savage,Elizabeth,18,1
1,2,Camacho,Eddie,15,2
2,3,Kennedy,Amanda,39,2
3,4,Wilson,Keith,15,3
4,5,Holder,Kristy,11,2
...,...,...,...,...,...
95,96,Anderson,Richard,27,4
96,97,Robinson,Sabrina,39,4
97,98,Ortiz,Jonathan,23,2
98,99,Peters,Alan,40,5


In [106]:
H_specialite = H_specialite.rename(columns={'Spécialité':'specialite'})
H_specialite

,id_specialite,specialite
0,1,Généraliste
1,2,Cardiologie
2,3,Radiologie
3,4,Pédiatrie
4,5,Dermatologie


## Dimension services

In [110]:
H_type_service = services[['Type_Service']].drop_duplicates().reset_index(drop=True)
H_type_service['id_type_service'] = H_type_service.index + 1
H_type_service

,Type_Service,id_type_service
0,Urgence,1
1,Consultation,2
2,Chirurgie,3


In [ ]:
D_service = services[['ID_Service','Type_Service','Nom_Service','Responsable']]
D_service = D_service.merge(H_type_service[['Type_Service','id_type_service']], on='Type_Service')
D_service = D_service.drop(columns=['Type_Service'])

D_service


,ID_Service,Nom_Service,Responsable,id_type_service
0,1,Service C1,Dr. Smith,1
1,2,Service D2,Dr. Le,1
2,3,Service B3,Dr. Simmons,2
3,4,Service E4,Dr. Olsen,2
4,5,Service A5,Dr. Taylor,3
5,6,Service C6,Dr. Mccoy,2
6,7,Service C7,Dr. Collins,1
7,8,Service D8,Dr. Flores,3
8,9,Service B9,Dr. Andrews,2
9,10,Service D10,Dr. Evans,2


In [111]:
Service = D_service.rename(columns={
    'ID_Service':'id_service',
    'Nom_Service':'nom_service',
    'Responsable':'responsable'
})
Service

,id_service,nom_service,responsable,id_type_service
0,1,Service C1,Dr. Smith,1
1,2,Service D2,Dr. Le,1
2,3,Service B3,Dr. Simmons,2
3,4,Service E4,Dr. Olsen,2
4,5,Service A5,Dr. Taylor,3
5,6,Service C6,Dr. Mccoy,2
6,7,Service C7,Dr. Collins,1
7,8,Service D8,Dr. Flores,3
8,9,Service B9,Dr. Andrews,2
9,10,Service D10,Dr. Evans,2


In [113]:
H_type_service = H_type_service.rename(columns={'Type_Service':'type_service'})
H_type_service

,type_service,id_type_service
0,Urgence,1
1,Consultation,2
2,Chirurgie,3


## Dimension Traitement

In [145]:
test =consultations['Traitement'].drop_duplicates()
test

0        Médicaments contre la douleur
1                      Plâtre et suivi
2                    Antihistaminiques
3         Prescription d'antibiotiques
4                Inhalateurs prescrits
6                   Régime et insuline
13    Prescription d'antihypertenseurs
Name: Traitement, dtype: object

In [146]:
D_traitement = consultations[['Traitement']].drop_duplicates().reset_index(drop=True)

In [147]:
def cat_traitement(t):

    if "antihypertenseurs" in t:
        return "Traitement médicamenteux"
    
    elif "Médicaments" in t or "Antihistaminiques" in t:
        return "Traitement médicamenteux"
    
    elif "antibiotiques" in t:
        return "Traitement antibiotique"
    
    elif "Plâtre" in t:
        return "Traitement orthopédique"
    
    elif "Inhalateurs" in t:
        return "Traitement respiratoire"
    
    elif "insuline" in t:
        return "Traitement diabétique"
    
    else:
        return "Autre"

In [148]:
D_traitement['cat_traitement'] = D_traitement['Traitement'].apply(cat_traitement)
D_traitement['cat_traitement']

0    Traitement médicamenteux
1     Traitement orthopédique
2    Traitement médicamenteux
3     Traitement antibiotique
4     Traitement respiratoire
5       Traitement diabétique
6    Traitement médicamenteux
Name: cat_traitement, dtype: object

In [149]:
H_cat_traitement = D_traitement[['cat_traitement']].drop_duplicates().reset_index(drop=True)

H_cat_traitement['id_cat_traitement'] = H_cat_traitement.index + 1

H_cat_traitement = H_cat_traitement[['id_cat_traitement','cat_traitement']]
H_cat_traitement

,id_cat_traitement,cat_traitement
0,1,Traitement médicamenteux
1,2,Traitement orthopédique
2,3,Traitement antibiotique
3,4,Traitement respiratoire
4,5,Traitement diabétique


In [ ]:
D_traitement = D_traitement.merge(H_cat_traitement,on='cat_traitement')
D_traitement['id_traitement'] = D_traitement.index + 1
D_traitement = D_traitement[['id_traitement','Traitement','id_cat_traitement']]


In [151]:
D_traitement

,id_traitement,Traitement,id_cat_traitement
0,1,Médicaments contre la douleur,1
1,2,Plâtre et suivi,2
2,3,Antihistaminiques,1
3,4,Prescription d'antibiotiques,3
4,5,Inhalateurs prescrits,4
5,6,Régime et insuline,5
6,7,Prescription d'antihypertenseurs,1


## Dimension Diagnostic 

In [153]:
D_diagnostic = consultations[['Diagnostic']].drop_duplicates().reset_index(drop=True)
D_diagnostic

,Diagnostic
0,Diabète
1,Hypertension
2,Asthme
3,Allergie saisonnière
4,Migraine
5,Infection respiratoire
6,Fracture osseuse


In [154]:
def cat_diagnostic(d):

    if d == "Diabète":
        return "Maladie métabolique"
    
    elif d == "Hypertension":
        return "Maladie cardiovasculaire"
    
    elif d in ["Asthme","Infection respiratoire"]:
        return "Maladie respiratoire"
    
    elif d == "Allergie saisonnière":
        return "Maladie immunitaire"
    
    elif d == "Migraine":
        return "Maladie neurologique"
    
    elif d == "Fracture osseuse":
        return "Traumatisme orthopédique"
    
    else:
        return "Autre"

In [155]:
D_diagnostic['cat_diagnostic'] = D_diagnostic['Diagnostic'].apply(cat_diagnostic)

In [156]:
H_cat_diagnostic = D_diagnostic[['cat_diagnostic']].drop_duplicates().reset_index(drop=True)

H_cat_diagnostic['id_cat_diagnostic'] = H_cat_diagnostic.index + 1

H_cat_diagnostic = H_cat_diagnostic[['id_cat_diagnostic','cat_diagnostic']]

In [158]:
H_cat_diagnostic

,id_cat_diagnostic,cat_diagnostic
0,1,Maladie métabolique
1,2,Maladie cardiovasculaire
2,3,Maladie respiratoire
3,4,Maladie immunitaire
4,5,Maladie neurologique
5,6,Traumatisme orthopédique


In [157]:
D_diagnostic = D_diagnostic.merge(H_cat_diagnostic,on='cat_diagnostic')
D_diagnostic['id_diagnostic'] = D_diagnostic.index + 1
D_diagnostic = D_diagnostic[['id_diagnostic','Diagnostic','id_cat_diagnostic']]
D_diagnostic

,id_diagnostic,Diagnostic,id_cat_diagnostic
0,1,Diabète,1
1,2,Hypertension,2
2,3,Asthme,3
3,4,Allergie saisonnière,4
4,5,Migraine,5
5,6,Infection respiratoire,3
6,7,Fracture osseuse,6


## Fait : consultation

In [159]:
F_consultation = consultations.copy()

In [160]:
F_consultation = F_consultation.merge(
    D_diagnostic[['Diagnostic','id_diagnostic']],
    on='Diagnostic'
)

In [161]:
F_consultation = F_consultation.merge(
    D_traitement[['Traitement','id_traitement']],
    on='Traitement'
)

In [162]:
F_consultation = F_consultation.drop(columns=['Diagnostic','Traitement'])

In [163]:
F_consultation = F_consultation.rename(columns={
    'ID_Consultation':'id_consultation',
    'ID_Patient':'id_patient',
    'ID_Docteur':'id_docteur',
    'ID_Service':'id_service',
    'ID_Temps':'id_temps',
    'Coût':'cout',
    'Durée':'duree'
})

In [164]:
F_consultation['nb_consultation'] = 1

In [165]:
F_consultation = F_consultation[[
'id_consultation',
'id_patient',
'id_docteur',
'id_temps',
'id_service',
'id_diagnostic',
'id_traitement',
'duree',
'cout',
'nb_consultation'
]]

In [166]:
F_consultation

,id_consultation,id_patient,id_docteur,id_temps,id_service,id_diagnostic,id_traitement,duree,cout,nb_consultation
0,1,277,49,259,8,1,1,100,198.09,1
1,2,56,39,69,5,1,2,12,428.25,1
2,3,252,62,150,1,2,3,32,143.03,1
3,4,84,26,17,9,3,4,88,491.38,1
4,5,988,28,178,2,4,5,26,56.27,1
...,...,...,...,...,...,...,...,...,...,...
9995,9996,585,5,197,6,1,2,78,223.98,1
9996,9997,607,62,340,9,1,2,11,247.13,1
9997,9998,751,29,51,4,1,1,103,115.18,1
9998,9999,366,24,118,5,2,5,70,248.11,1


## EXPORT

In [168]:
D_date.to_csv("export/D_date.csv", index=False)
D_patient.to_csv("export/D_patient.csv", index=False)
D_doctor.to_csv("export/D_doctor.csv", index=False)
Service.to_csv("export/D_service.csv", index=False)
D_diagnostic.to_csv("export/D_diagnostic.csv", index=False)
D_traitement.to_csv("export/D_traitement.csv", index=False)

In [169]:
H_annee.to_csv("export/H_annee.csv", index=False)
H_trimestre.to_csv("export/H_trimestre.csv", index=False)
H_mois.to_csv("export/H_mois.csv", index=False)

H_ville.to_csv("export/H_ville.csv", index=False)
H_code_postal.to_csv("export/H_code_postal.csv", index=False)

H_specialite.to_csv("export/H_specialite.csv", index=False)
H_type_service.to_csv("export/H_type_service.csv", index=False)

H_cat_diagnostic.to_csv("export/H_cat_diagnostic.csv", index=False)
H_cat_traitement.to_csv("export/H_cat_traitement.csv", index=False)

In [170]:
F_consultation.to_csv("export/F_consultation.csv", index=False)